# Server Failure Prediction — From Data to a Trained PyTorch Model
**Goal:** predict whether a server will fail in the next 24 hours.

Synthetic learning project:
**data → tensors → train/test → model → logits → loss → backprop → optimizer → evaluation → inference**

Features: temperature, power, memory errors, fan speed, age.  
Label: `0` = no failure, `1` = failure.

> Data is synthetic; it is not production AWS/Microsoft telemetry.

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 2. Create synthetic server data
We create 10,000 servers with 5 features, so `X.shape = [10000,5]`.
There is one binary answer per server, so `y.shape = [10000,1]`.

In [ ]:
N = 10_000
temperature = 55 + 45 * torch.rand(N,1)
power = 500 + 500 * torch.rand(N,1)
memory_errors = 60 * torch.rand(N,1)
fan_speed = 3500 + 3500 * torch.rand(N,1)
age = 5 * torch.rand(N,1)

X_raw = torch.cat([temperature,power,memory_errors,fan_speed,age], dim=1)

feature_mean = X_raw.mean(dim=0, keepdim=True)
feature_std = X_raw.std(dim=0, keepdim=True)
X = (X_raw-feature_mean)/feature_std

# Synthetic hidden rule used only to create teaching labels.
risk_score = 1.2*X[:,0] + 0.7*X[:,1] + 1.6*X[:,2] + 0.3*X[:,3] + 0.8*X[:,4] - 0.4
failure_probability = torch.sigmoid(risk_score)
y = torch.bernoulli(failure_probability).unsqueeze(1)

print("X:", X.shape, "y:", y.shape)
print("Failure rate:", round(y.mean().item(),3))

## 3. 80/20 train-test split

In [ ]:
indices = torch.randperm(N)
train_size = int(0.8*N)
train_idx, test_idx = indices[:train_size], indices[train_size:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

## 4. Build the model
`nn.Linear(5,1)` = five input features → five learnable weights + one bias → one raw output (logit).

In [ ]:
model = nn.Linear(5,1)
print(model)
print("weights:", model.weight.shape, "bias:", model.bias.shape)

## 5. Loss and optimizer
`BCEWithLogitsLoss` compares binary targets with raw logits.  
SGD uses gradients to update the learnable parameters.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

## 6. Train
1. clear old gradients  
2. forward pass  
3. calculate loss  
4. backpropagate gradients  
5. update weights

In [ ]:
loss_history=[]
for epoch in range(101):
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss {loss.item():.4f}")

## 7. Watch the loss change

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training Loss Over Time")
plt.show()

## 8. Evaluate on unseen data
Evaluation does **not** update weights. We convert logits to probabilities with sigmoid and use 0.5 as a simple threshold.

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test)
    probs = torch.sigmoid(test_logits)
    pred = (probs >= 0.5).float()

accuracy = (pred == y_test).float().mean().item()
tp = ((pred==1)&(y_test==1)).sum().item()
fp = ((pred==1)&(y_test==0)).sum().item()
fn = ((pred==0)&(y_test==1)).sum().item()
tn = ((pred==0)&(y_test==0)).sum().item()
precision = tp/(tp+fp) if tp+fp else 0
recall = tp/(tp+fn) if tp+fn else 0
f1 = 2*precision*recall/(precision+recall) if precision+recall else 0

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1       : {f1:.3f}")
print("TN, FP, FN, TP:", tn, fp, fn, tp)

## 9. Inference on one new server

In [ ]:
new_raw = torch.tensor([[92.,940.,42.,6400.,4.]], dtype=torch.float32)
new_x = (new_raw-feature_mean)/feature_std

with torch.no_grad():
    new_logit = model(new_x)
    new_probability = torch.sigmoid(new_logit)

print("Input shape:", new_x.shape)
print("Logit shape:", new_logit.shape)
print(f"Predicted failure probability: {new_probability.item():.3f}")

## 10. End-to-end mental model
`raw data → X/y → train/test → nn.Linear → logits → BCE loss → backward → gradients → SGD → updated weights → epochs → trained model → test → inference`

### Next
Move the same workload to a **GPU**:
**CPU → GPU memory → CUDA execution → utilization → memory → throughput**